In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve as scipy_convolve
from matplotlib.widgets import Slider

# --- 1. Definir las funciones f(x) y g(x) ---
x = np.linspace(-10, 10, 500)
dx = x[1] - x[0]

def gaussian(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma)**2)

def rect(x, center, width):
    return np.where(np.abs(x - center) <= width / 2, 1.0, 0.0)

f = gaussian(x, mu=-3, sigma=1.5)
g_original = rect(x, center=0, width=2)
g_original = g_original / np.sum(g_original * dx) # Normalizar

In [ ]:
import numpy as np
from ipywidgets import interact, FloatSlider, Dropdown
from IPython.display import display, Markdown

plt.style.use("seaborn-v0_8-poster")
plt.rcParams["figure.figsize"] = (10, 6)

# Introducción teórica
## Convolución entre dos funciones

La **convolución** de dos funciones continuas $f(t)$ y $g(t)$ se define como:

$$ (f * g)(t) = \int_{-\infty}^{\infty} f(\tau)\, g(t - \tau)\, d\tau $$

Intuitivamente, esto representa el *área de superposición* entre una función $f$ y una versión invertida y desplazada de $g$.

La convolución aparece en múltiples campos:
- Procesamiento de señales.
- Análisis de sistemas lineales invariantes en el tiempo.
- Física (difusión, respuesta de sistemas, óptica, etc.).

In [ ]:
# Funciones base
def rectangular(t, width=1.0):
    return np.where(np.abs(t) <= width / 2, 1.0, 0.0)

def gaussiana(t, sigma=0.5):
    return np.exp(-t**2 / (2 * sigma**2))

def triangular(t, width=1.0):
    return np.where(np.abs(t) <= width, 1 - np.abs(t) / width, 0)

# Selector de funciones
funciones = {
    "Rectangular": rectangular,
    "Gaussiana": gaussiana,
    "Triangular": triangular
}

In [ ]:
# Visualización interactiva
def visualizar_convolucion(func1_name, func2_name, width1, width2, desplazamiento):
    t = np.linspace(-5, 5, 1000)

    f = funciones[func1_name](t, width1)
    g = funciones[func2_name](t, width2)

    # Invertir y desplazar g
    g_inv = funciones[func2_name](-t + desplazamiento, width2)

    # Calcular convolución numérica
    dt = t[1] - t[0]
    conv = np.convolve(f, g, mode='same') * dt

    # Gráficas
    plt.figure(figsize=(12, 8))

    plt.subplot(3, 1, 1)
    plt.plot(t, f, label=f"$f(t)$: {func1_name}")
    plt.plot(t, g_inv, label=f"$g(t - τ)$ desplazada")
    plt.fill_between(t, f * g_inv, alpha=0.3, label="Producto puntual")
    plt.title("Superposición y producto f(τ)·g(t−τ)")
    plt.legend()
    plt.grid(True)

    plt.subplot(3, 1, 2)
    plt.plot(t, conv, label="Convolución (f*g)(t)")
    plt.axvline(desplazamiento, color='r', linestyle='--', alpha=0.5)
    plt.title("Resultado de la convolución")
    plt.legend()
    plt.grid(True)

    plt.subplot(3, 1, 3)
    plt.plot(t, f, label="f(t)")
    plt.plot(t, g, label="g(t)")
    plt.title("Funciones originales")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# --- Widgets interactivos ---
interact(
    visualizar_convolucion,
    func1_name=Dropdown(options=funciones.keys(), value="Rectangular", description="f(t):"),
    func2_name=Dropdown(options=funciones.keys(), value="Gaussiana", description="g(t):"),
    width1=FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description="Ancho f"),
    width2=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description="Ancho g"),
    desplazamiento=FloatSlider(value=0.0, min=-5, max=5, step=0.1, description="Desplazamiento"),
);

interactive(children=(Dropdown(description='f(t):', options=('Rectangular', 'Gaussiana', 'Triangular'), value=…

In [ ]:
# ============================================================
# Notebook interactivo: Visualización de la convolución de dos funciones
# Autor: Carlos Andrés Gómez Vasco
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, Output, VBox
from IPython.display import display, Markdown, clear_output

plt.style.use("seaborn-v0_8-poster")
plt.rcParams["figure.figsize"] = (10, 6)

# --- Introducción teórica ---
display(Markdown(r"""
# Convolución entre dos funciones

La **convolución** de dos funciones continuas \( f(t) \) y \( g(t) \) se define como:

\[
(f * g)(t) = \int_{-\infty}^{\infty} f(\tau)\, g(t - \tau)\, d\tau
\]

Intuitivamente, esto representa el *área de superposición* entre una función \( f \)
y una versión invertida y desplazada de \( g \).

La convolución aparece en múltiples campos:
- Procesamiento de señales.
- Análisis de sistemas lineales invariantes en el tiempo.
- Física (difusión, respuesta de sistemas, óptica, etc.).
"""))

# --- Definición de funciones base ---
def rectangular(t, width=1.0):
    return np.where(np.abs(t) <= width / 2, 1.0, 0.0)

def gaussiana(t, sigma=0.5):
    return np.exp(-t**2 / (2 * sigma**2))

def triangular(t, width=1.0):
    return np.where(np.abs(t) <= width, 1 - np.abs(t) / width, 0)

funciones = {
    "Rectangular": rectangular,
    "Gaussiana": gaussiana,
    "Triangular": triangular
}

# --- Zona de texto interactiva ---
out_texto = Output()

def visualizar_convolucion(func1_name, func2_name, width1, width2, desplazamiento):
    t = np.linspace(-5, 5, 500)
    dt = t[1] - t[0]

    f = funciones[func1_name](t, width1)
    g = funciones[func2_name](t, width2)

    # g invertida y desplazada
    g_inv = funciones[func2_name](-t + desplazamiento, width2)

    # Producto puntual
    producto = f * g_inv

    # Convolución numérica (discreta)
    conv = np.convolve(f, g, mode='same') * dt

    # Gráficas
    plt.figure(figsize=(12, 9))

    plt.subplot(3, 1, 1)
    plt.plot(t, f, label=f"$f(\\tau)$: {func1_name}")
    plt.plot(t, g_inv, label=f"$g(t - \\tau)$ desplazada")
    plt.fill_between(t, producto, alpha=0.3, color='orange', label="Producto puntual")
    plt.title("Superposición y producto $f(\\tau)·g(t−\\tau)$")
    plt.legend()
    plt.grid(True)

    plt.subplot(3, 1, 2)
    plt.plot(t, conv, label="Convolución $(f*g)(t)$", color="purple")
    plt.axvline(desplazamiento, color='r', linestyle='--', alpha=0.6, label=f"t = {desplazamiento:.2f}")
    plt.title("Resultado de la convolución")
    plt.legend()
    plt.grid(True)

    plt.subplot(3, 1, 3)
    plt.plot(t, f, label="$f(t)$")
    plt.plot(t, g, label="$g(t)$")
    plt.title("Funciones originales")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    # --- Mostrar los valores paso a paso ---
    with out_texto:
        clear_output(wait=True)
        idx = (np.abs(t - 0)).argmin()
        f_valores = f[:10]
        g_valores = g_inv[:10]
        productos = producto[:10]
        integral_aprox = np.sum(producto) * dt

        display(Markdown(f"""
### 🧮 Paso a paso del cálculo en t = {desplazamiento:.2f}

| τ | f(τ) | g(t−τ) | Producto f(τ)·g(t−τ) |
|:--:|:--:|:--:|:--:|
""" + "\n".join([
f"| {t[i]:.2f} | {f[i]:.3f} | {g_inv[i]:.3f} | {producto[i]:.3f} |"
for i in range(0, len(t), len(t)//10)
]) + f"""

**Integral aproximada (área bajo la curva):**
(f * g)({desplazamiento:.2f}) $\\approx$ {integral_aprox:.4f}
"""))

# --- Widgets interactivos ---
ui = interact(
    visualizar_convolucion,
    func1_name=Dropdown(options=funciones.keys(), value="Rectangular", description="f(t):"),
    func2_name=Dropdown(options=funciones.keys(), value="Gaussiana", description="g(t):"),
    width1=FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description="Ancho f"),
    width2=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description="Ancho g"),
    desplazamiento=FloatSlider(value=0.0, min=-5, max=5, step=0.2, description="t (desplazamiento)")
);

display(out_texto)



# Convolución entre dos funciones

La **convolución** de dos funciones continuas \( f(t) \) y \( g(t) \) se define como:

\[
(f * g)(t) = \int_{-\infty}^{\infty} f(\tau)\, g(t - \tau)\, d\tau
\]

Intuitivamente, esto representa el *área de superposición* entre una función \( f \)
y una versión invertida y desplazada de \( g \).

La convolución aparece en múltiples campos:
- Procesamiento de señales.
- Análisis de sistemas lineales invariantes en el tiempo.
- Física (difusión, respuesta de sistemas, óptica, etc.).


interactive(children=(Dropdown(description='f(t):', options=('Rectangular', 'Gaussiana', 'Triangular'), value=…

Output()